# LMIC-CRIS: prostate cancer risk model, final reproducible analysis

Input: `df.xlsx` (the de-identified dataset) (sheet `Cleaning`, 346 raw records, patient names replaced by `Patient_ID`).
All steps are deterministic (seed 42). Outputs: `results_summary.csv`, `pending_records.csv`, `record_level_audit.csv` (Excel row numbers only, no names).

Method summary: exact duplicates removed, records with invalid PSA, date or age removed, outcome classified from the primary diagnosis line (advisory NB notes are not used for cancer or benign keywords), records without definitive histology excluded from the primary cohort, XGBoost evaluated by 5-fold stratified cross-validation with native missing-value handling and no imputation. A one-record-per-patient analysis and a bounding sensitivity analysis are reported.

In [1]:
import pandas as pd, numpy as np, re, sys
import sklearn, xgboost
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

RNG = 42
SRC = "df.xlsx"
print("python", sys.version.split()[0], "| xgboost", xgboost.__version__, "| scikit-learn", sklearn.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__)

python 3.12.3 | xgboost 3.4.1 | scikit-learn 1.8.0 | pandas 3.0.2 | numpy 2.4.4


## 1. Load and remove exact duplicates
Source row = Excel row number (header = row 1). Duplicates are identical across all columns except the running index `Unnamed: 0`.

In [2]:
raw = pd.read_excel(SRC, sheet_name="Cleaning")
raw["source_row"] = np.arange(len(raw)) + 2
print(f"STEP 1  raw records: {len(raw)}")

cols = [c for c in raw.columns if c not in ("Unnamed: 0", "source_row")]
dup = raw.duplicated(subset=cols, keep="first")
df = raw[~dup].copy()
print(f"STEP 2  exact duplicate records removed: {int(dup.sum())}  -> {len(df)}")

STEP 1  raw records: 346


STEP 2  exact duplicate records removed: 74  -> 272


## 2. Parse values and apply the validity filter
A record is valid if both PSA values and both PSA dates parse, PSA date 1 is on or after 2000-01-01 (earlier dates, such as 1970-01-01, are spreadsheet placeholders), PSA date 2 is after PSA date 1, and age is between 30 and 110. PSA values are capped at 100 ng/mL.

In [3]:
def clean_psa(v):
    if pd.isna(v): return np.nan
    v = str(v).lower().replace("ng/ml", "").replace("ng/mls", "").replace(">", "").strip()
    try: return float(v)
    except Exception: return np.nan

df["AGE_n"] = pd.to_numeric(df["AGE"].astype(str).str.extract(r"(\d+)", expand=False), errors="coerce")
df["psa1"] = df["PSA_Value_1"].apply(clean_psa).clip(upper=100)
df["psa2"] = df["PSA_Value_2"].apply(clean_psa).clip(upper=100)
df["d1"] = pd.to_datetime(df["PSA_Date_1"], errors="coerce")
df["d2"] = pd.to_datetime(df["PSA_Date_2"], errors="coerce")
ok = (df["psa1"].notna() & df["psa2"].notna() & df["d1"].notna() & df["d2"].notna()
      & (df["d1"] >= "2000-01-01") & (df["d2"] > df["d1"]) & df["AGE_n"].between(30, 110))
print(f"STEP 3  invalid PSA/date/age removed: {int((~ok).sum())}  -> {int(ok.sum())}")
df["valid"] = ok
valid = df[ok].copy()

STEP 3  invalid PSA/date/age removed: 69  -> 203


## 3. Outcome classification
Cancer and benign keywords are read from the primary diagnosis line only (text before any `NB` note). Adequacy and uncertainty wording is read from the full text, so an inadequate specimen stays excluded even when the note sits in the `NB` section. Records that cannot be classified definitively are excluded from the primary cohort and listed in `pending_records.csv` for clinical review.

In [4]:
def classify(diag):
    full = str(diag).upper()
    s = re.split(r"\bN\.?B\b", full)[0]
    malignant = ("ADENOCARCINOMA" in s or "CARCINOMA" in s or "MALIGN" in s or "SARCOMA" in s)
    benign = ("HYPERPLASIA" in s or "PROSTATITIS" in s or "BENIGN" in s)
    uncertain = bool(re.search(
        r"READ TEXT|INADEQUATE|INSUFFICIENT|NOT REPRESENTATIVE|COULD NOT BE MADE|CANNOT BE RULED|^\s*\?|INTRA-?EPITHE|SUSPICIOUS", s)) \
        or bool(re.search(r"INADEQUATE|INSUFFICIENT|NOT REPRESENTATIVE|COULD NOT BE MADE", full))
    if uncertain and not (malignant and "ADENOCARCINOMA" in s and "GLEASON" in s):
        return "no_definitive_histology"
    if "?" in s:
        return "no_definitive_histology"
    if malignant: return "malignant"
    if benign: return "benign"
    return "no_definitive_histology"

valid["cls"] = valid["Pathological Diagnosis"].apply(classify)
pend = valid[valid["cls"] == "no_definitive_histology"]
print(f"STEP 4  no definitive histology, excluded from primary cohort: {len(pend)}")
print("        Excel rows:", sorted(pend["source_row"].tolist()))
pend[["source_row", "Pathological Diagnosis"]].to_csv("pending_records.csv", index=False)

STEP 4  no definitive histology, excluded from primary cohort: 21
        Excel rows: [10, 108, 121, 131, 158, 181, 208, 211, 269, 272, 276, 284, 292, 295, 313, 323, 331, 332, 334, 335, 345]


## 4. Features
PSA velocity (ng/mL/yr) and doubling time from the two PSA readings. Prostate volume is extracted from free-text imaging findings and counted only when an explicit mL or cm3 unit is present (cm2 is an area, not a volume). Missing volumes are left missing: XGBoost handles them natively, with no imputation.

In [5]:
df = valid
df["time_yr"] = (df["d2"] - df["d1"]).dt.days / 365.25
df["vel"] = (df["psa2"] - df["psa1"]) / df["time_yr"]
df["dbl"] = np.where(df["vel"] > 0, 0.693 / (df["vel"] / df["psa1"]), np.nan)
ML_RE = re.compile(r"(\d+\.?\d*)\s?(mls|ml|cm3|cm³)\b", re.I)
df["vol"] = df["Imaging_Findings"].apply(
    lambda t: float(ML_RE.search(str(t)).group(1)) if pd.notna(t) and ML_RE.search(str(t)) else np.nan)
df["imaging_susp"] = df["Imaging_Findings"].str.contains(
    "enlarged|lesion|nodule|hypoechoic|remarkable|enlarge", case=False, na=False).astype(int)
df["HTN"] = df["Comorbidities"].str.contains("HTN|HHDx", case=False, na=False).astype(int)
df["T2DM"] = df["Comorbidities"].str.contains("T2DM|DM|diabetes", case=False, na=False).astype(int)
FEATS = ["AGE_n", "psa1", "psa2", "vel", "dbl", "vol", "imaging_susp", "HTN", "T2DM"]

prim = df[df["cls"] != "no_definitive_histology"]
vol_miss = prim["vol"].isna().sum()
print(f"STEP 5  primary cohort: {len(prim)}  "
      f"({(prim['cls']=='malignant').sum()} malignant / {(prim['cls']=='benign').sum()} benign)")
print(f"        prostate volume missing (mL/cm3 only): {vol_miss}/{len(prim)} = {100*vol_miss/len(prim):.1f}%")

STEP 5  primary cohort: 182  (107 malignant / 75 benign)
        prostate volume missing (mL/cm3 only): 16/182 = 8.8%


## 5. Evaluation
5-fold stratified cross-validation, seed 42. The one-record-per-patient analysis keeps the first record per `Patient_ID` and is the conservative headline, because several patients have more than one record and record-level folds can place the same patient in both training and test data.

In [6]:
def cv_auc(sub, label, rows):
    X = sub[FEATS].astype(float).values
    y = (sub["cls_final"] == "malignant").astype(int).values
    spw = (y == 0).sum() / (y == 1).sum()
    m = XGBClassifier(random_state=RNG, learning_rate=0.1, max_depth=3,
                      n_estimators=50, scale_pos_weight=spw, eval_metric="logloss")
    s = cross_val_score(m, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=RNG), scoring="roc_auc")
    print(f"{label:52s} n={len(sub):3d} ({int(y.sum())}/{int((y==0).sum())})  CV AUC {s.mean():.3f} ± {s.std():.3f}")
    rows.append({"analysis": label.strip(), "n": len(sub), "malignant": int(y.sum()),
                 "benign": int((y == 0).sum()), "cv_auc_mean": round(s.mean(), 3), "cv_auc_sd": round(s.std(), 3)})

rows = []
print("\n==== RESULTS (5-fold stratified CV, seed 42) ====")
for label, assign in [("PRIMARY: non-definitive excluded", None),
                      ("BOUND: non-definitive as malignant", "malignant"),
                      ("BOUND: non-definitive as benign", "benign")]:
    d = df.copy()
    d["cls_final"] = d["cls"]
    if assign is None:
        d = d[d["cls"] != "no_definitive_histology"]
    else:
        d.loc[d["cls"] == "no_definitive_histology", "cls_final"] = assign
    cv_auc(d, label, rows)
    d1 = d[~d.duplicated(subset=["Patient_ID"], keep="first")]
    cv_auc(d1, "  one record per patient", rows)

pd.DataFrame(rows).to_csv("results_summary.csv", index=False)
print("\nwritten: results_summary.csv, pending_records.csv")


==== RESULTS (5-fold stratified CV, seed 42) ====
PRIMARY: non-definitive excluded                     n=182 (107/75)  CV AUC 0.837 ± 0.067


  one record per patient                             n=139 (77/62)  CV AUC 0.751 ± 0.105


BOUND: non-definitive as malignant                   n=203 (128/75)  CV AUC 0.808 ± 0.081
  one record per patient                             n=157 (95/62)  CV AUC 0.716 ± 0.120


BOUND: non-definitive as benign                      n=203 (107/96)  CV AUC 0.851 ± 0.059


  one record per patient                             n=157 (77/80)  CV AUC 0.787 ± 0.065

written: results_summary.csv, pending_records.csv


## 6. Record-level audit (no names)
One row per raw record, showing why each record is in or out of the analysis.

In [7]:
aud = raw[["source_row", "Imaging_Type", "Imaging_Findings"]].copy()
aud["duplicate_removed"] = dup.values
aud = aud.merge(df[["source_row", "valid", "cls", "vol", "Patient_ID"]], on="source_row", how="left")
inv = raw.loc[~dup.values, ["source_row"]].merge(df[["source_row", "valid"]], on="source_row", how="left")
aud["valid"] = aud["valid"].astype("boolean")
def stage(r):
    if r.duplicate_removed: return "removed: exact duplicate"
    if r.valid is False or pd.isna(r.valid): return "removed: invalid PSA/date/age"
    if r.cls == "no_definitive_histology": return "excluded: no definitive histology (clinical review)"
    return "in primary cohort"
aud["status"] = aud.apply(stage, axis=1)
aud["class_used"] = np.where(aud["status"] == "in primary cohort", aud["cls"], "")
aud["volume_ml"] = aud["vol"]
aud["volume_status"] = np.where(aud["status"] == "in primary cohort",
                                np.where(aud["vol"].isna(), "missing", "observed"), "")
first_of_patient = prim.drop_duplicates(subset=["Patient_ID"], keep="first")["source_row"]
aud["in_patient_level_analysis"] = aud["source_row"].isin(first_of_patient)
aud = aud.drop(columns=["valid", "cls", "vol", "Patient_ID", "duplicate_removed"])
aud.to_csv("record_level_audit.csv", index=False)
print(aud["status"].value_counts().to_string())
print("patient-level records:", int(aud["in_patient_level_analysis"].sum()))

status
in primary cohort                                      182
removed: exact duplicate                                74
removed: invalid PSA/date/age                           69
excluded: no definitive histology (clinical review)     21
patient-level records: 139


## 7. Referral threshold analysis (patient level)
Out-of-fold predicted probabilities from the same 5-fold cross-validation, one record per patient. For each sensitivity target, the highest probability threshold that still reaches that sensitivity is used, and the table shows what that threshold would do to 100 patients like these. Thresholds are chosen on the same out-of-fold predictions that are scored, so these figures are optimistic and are descriptive only. They are not an estimate of clinical performance or of system effect.

In [8]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score

pt = df[df["cls"] != "no_definitive_histology"]
pt = pt[~pt.duplicated(subset=["Patient_ID"], keep="first")]
X = pt[FEATS].astype(float).values
y = (pt["cls"] == "malignant").astype(int).values
m = XGBClassifier(random_state=RNG, learning_rate=0.1, max_depth=3, n_estimators=50,
                  scale_pos_weight=(y == 0).sum() / (y == 1).sum(), eval_metric="logloss")
proba = cross_val_predict(m, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=RNG), method="predict_proba")[:, 1]
print(f"patients: {len(y)} ({y.sum()} malignant / {(y==0).sum()} benign) | pooled out-of-fold AUC {roc_auc_score(y, proba):.3f}")

rows = []
for target in (0.95, 0.90, 0.80):
    thr = max(t for t in np.unique(proba) if ((proba >= t) & (y == 1)).sum() / (y == 1).sum() >= target)
    pred = proba >= thr
    tp = int((pred & (y == 1)).sum()); fn = int((~pred & (y == 1)).sum())
    fp = int((pred & (y == 0)).sum()); tn = int((~pred & (y == 0)).sum())
    rows.append({"target_sensitivity": target, "threshold": round(float(thr), 3),
                 "sensitivity": round(tp / (tp + fn), 3), "specificity": round(tn / (tn + fp), 3),
                 "referred_per_100": round(100 * (tp + fp) / len(y), 1),
                 "benign_not_referred_per_100": round(100 * tn / len(y), 1),
                 "cancers_missed_per_100": round(100 * fn / len(y), 1),
                 "tp": tp, "fn": fn, "fp": fp, "tn": tn})
thr_df = pd.DataFrame(rows)
thr_df.to_csv("threshold_analysis.csv", index=False)
print(thr_df.to_string(index=False))
print("baseline: referring everyone = 100 referred, ", round(100 * (y == 0).sum() / len(y), 1), "benign per 100 referred unnecessarily")

patients: 139 (77 malignant / 62 benign) | pooled out-of-fold AUC 0.745
 target_sensitivity  threshold  sensitivity  specificity  referred_per_100  benign_not_referred_per_100  cancers_missed_per_100  tp  fn  fp  tn
               0.95      0.065        0.961        0.048              95.7                          2.2                     2.2  74   3  59   3
               0.90      0.129        0.909        0.113              89.9                          5.0                     5.0  70   7  55   7
               0.80      0.292        0.805        0.468              68.3                         20.9                    10.8  62  15  33  29
baseline: referring everyone = 100 referred,  44.6 benign per 100 referred unnecessarily


## 8. BASELINE comparison and uncertainty (patient level)
AUC of single raw PSA readings as risk scores (no model fitting, same 139 patients), and a percentile bootstrap 95% interval (2,000 resamples, seed 42) for the pooled out-of-fold AUC of the model. Intervals describe sampling uncertainty only and do not reflect the optimism of selecting the model design on this cohort.

In [9]:
rng = np.random.default_rng(RNG)
def boot_ci(score, y, n=2000):
    aucs = []
    while len(aucs) < n:
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            aucs.append(roc_auc_score(y[i], score[i]))
    return np.percentile(aucs, [2.5, 97.5])

base = []
for name, score in [("Model (pooled out-of-fold)", proba),
                    ("PSA 1 alone", pt["psa1"].values),
                    ("PSA 2 alone", pt["psa2"].values),
                    ("PSA velocity alone (negated)", -pt["vel"].values)]:
    lo, hi = boot_ci(np.asarray(score, dtype=float), y)
    base.append({"score": name, "auc": round(roc_auc_score(y, score), 3), "ci_low": round(lo, 3), "ci_high": round(hi, 3)})
base_df = pd.DataFrame(base)
base_df.to_csv("baseline_comparison.csv", index=False)
print(base_df.to_string(index=False))
# PSA 4 ng/mL rule on the first reading, for reference
rule = (pt["psa1"].values >= 4)
print("PSA 1 >= 4 ng/mL: sensitivity", round(rule[y == 1].mean(), 3), "| specificity", round((~rule)[y == 0].mean(), 3))

                       score   auc  ci_low  ci_high
  Model (pooled out-of-fold) 0.745   0.661    0.826
                 PSA 1 alone 0.745   0.658    0.825
                 PSA 2 alone 0.659   0.569    0.756
PSA velocity alone (negated) 0.706   0.617    0.790
PSA 1 >= 4 ng/mL: sensitivity 0.909 | specificity 0.145
